In [ ]:
!pip install matplotlib

In [ ]:
import sqlite3

def inicializar_bd_avanzada():
    conexion = sqlite3.connect('clinica_local.db')
    cursor = conexion.cursor()

    # Tabla 1: Identificación Permanente del Paciente
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS pacientes (
        id_paciente INTEGER PRIMARY KEY AUTOINCREMENT,
        nombre_completo TEXT,
        curp TEXT,
        edad INTEGER,
        sexo TEXT,
        diagnosticos_previos TEXT
    )
    ''')

    # Tabla 2: Historial Clínico Dinámico (Basado en Expediente_Prueba.png)
    # Todos los campos médicos carecen de "NOT NULL" para permitir notas rápidas incompletas
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS registros_triage (
        id_registro INTEGER PRIMARY KEY AUTOINCREMENT,
        id_paciente INTEGER,
        fecha_registro TEXT,
        hora_registro TEXT,
        
        -- 2. Signos Vitales y Antropometría
        pa_sistolica INTEGER,
        pa_diastolica INTEGER,
        frec_cardiaca INTEGER,
        frec_respiratoria INTEGER,
        peso REAL,
        talla REAL,
        cintura REAL,
        glucosa INTEGER,
        temperatura REAL,
        spo2 INTEGER,
        
        -- 3. Contexto Clínico
        condicion_glucosa TEXT,
        embarazo_semanas INTEGER,
        toma_medicamentos TEXT,
        
        -- 4. y 5. Síntomas y Notas
        motivo_consulta TEXT,
        observaciones TEXT,
        
        FOREIGN KEY(id_paciente) REFERENCES pacientes(id_paciente)
    )
    ''')

    conexion.commit()
    conexion.close()
    print("✅ Base de datos de Triaje inicializada correctamente.")

# Ejecutar la función
inicializar_bd_avanzada()

In [ ]:
import base64
from PIL import Image
import matplotlib.pyplot as plt


ruta_prueba = "ficha_enfermeria.jpeg"

# Codificar a Base64
with open(ruta_prueba, "rb") as archivo_imagen:
    imagen_base64 = base64.b64encode(archivo_imagen.read()).decode('utf-8')

# Mostrar para la demo
imagen_pil = Image.open(ruta_prueba)
plt.imshow(imagen_pil)
plt.axis('off')
plt.show()

print("✅ Imagen 'ficha_enfermeria.jpeg' cargada y lista.")

In [ ]:
import requests
import json

url_local = "http://localhost:11434/api/generate"

prompt_maestro = """
Eres un asistente médico experto en extracción de datos. 
Analiza la imagen de este expediente clínico.
Extrae la información y devuelve ÚNICAMENTE un objeto JSON válido con esta estructura exacta.
REGLA DE ORO: Si un campo está en blanco, no se menciona o es ilegible en la imagen, asigna el valor null (sin comillas). No inventes datos.

Estructura requerida:
{
  "nombre_completo": "texto",
  "curp": "texto",
  "edad": numero,
  "sexo": "H o M",
  "fecha_registro": "DD/MM/AAAA",
  "hora_registro": "HH:MM",
  "pa_sistolica": numero,
  "pa_diastolica": numero,
  "frec_cardiaca": numero,
  "frec_respiratoria": numero,
  "peso": numero,
  "talla": numero,
  "cintura": numero,
  "glucosa": numero,
  "temperatura": numero,
  "spo2": numero,
  "condicion_glucosa": "Ayuno, Casual o null",
  "embarazo_semanas": numero,
  "diagnosticos_previos": "texto",
  "toma_medicamentos": "texto",
  "motivo_consulta": "texto",
  "observaciones": "texto"
}
"""

payload = {
    "model": "llava", 
    "prompt": prompt_maestro,
    "images": [imagen_base64], # ¡Reactivamos la inyección directa de la imagen!
    "stream": False,
    "format": "json" 
}

print("Analizando imagen nativamente con el modelo Multimodal... (GPU trabajando)")
respuesta = requests.post(url_local, json=payload)

if respuesta.status_code != 200:
    print(f"❌ Error: {respuesta.text}")
    resultado_crudo = ""
else:
    resultado_crudo = respuesta.json().get("response", "")
    print("✅ JSON extraído nativamente de los píxeles:\n", resultado_crudo)

In [ ]:
import sqlite3

if not resultado_crudo:
    print("⚠️ Operación abortada: No hay datos JSON para procesar.")
else:
    try:
        # Limpieza robusta del string
        texto_json = resultado_crudo.strip()
        if texto_json.startswith("```json"):
            texto_json = texto_json[7:-3].strip()
        elif texto_json.startswith("```"):
            texto_json = texto_json[3:-3].strip()
            
        datos = json.loads(texto_json)
        
        conexion = sqlite3.connect('clinica_local.db')
        cursor = conexion.cursor()
        
        # Inserción (Misma lógica que antes)
        nombre = datos.get('nombre_completo')
        curp = datos.get('curp')
        
        cursor.execute("SELECT id_paciente FROM pacientes WHERE nombre_completo = ? OR (curp = ? AND curp IS NOT NULL)", (nombre, curp))
        resultado_paciente = cursor.fetchone()
        
        if resultado_paciente:
            id_pac = resultado_paciente[0]
            print(f"🔄 Paciente existente encontrado (ID: {id_pac}).")
        else:
            cursor.execute('''INSERT INTO pacientes (nombre_completo, curp, edad, sexo, diagnosticos_previos) 
                              VALUES (?, ?, ?, ?, ?)''', 
                           (nombre, curp, datos.get('edad'), datos.get('sexo'), datos.get('diagnosticos_previos')))
            id_pac = cursor.lastrowid
            print(f"🆕 Nuevo paciente registrado (ID: {id_pac}).")
        
        cursor.execute('''
            INSERT INTO registros_triage (
                id_paciente, fecha_registro, hora_registro, pa_sistolica, pa_diastolica, 
                frec_cardiaca, frec_respiratoria, peso, talla, cintura, glucosa, temperatura, 
                spo2, condicion_glucosa, embarazo_semanas, toma_medicamentos, motivo_consulta, observaciones
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (
            id_pac, datos.get('fecha_registro'), datos.get('hora_registro'),
            datos.get('pa_sistolica'), datos.get('pa_diastolica'), datos.get('frec_cardiaca'),
            datos.get('frec_respiratoria'), datos.get('peso'), datos.get('talla'), datos.get('cintura'),
            datos.get('glucosa'), datos.get('temperatura'), datos.get('spo2'), datos.get('condicion_glucosa'),
            datos.get('embarazo_semanas'), datos.get('toma_medicamentos'), datos.get('motivo_consulta'),
            datos.get('observaciones')
        ))
        
        conexion.commit()
        conexion.close()
        
        print("✅ ¡Éxito! Expediente guardado.")
        
    except json.JSONDecodeError as e:
        print(f"❌ Error al parsear JSON: {e}")
        print(f"Esto fue lo que intentó leer:\n{texto_json}")
    except Exception as e:
        print(f"❌ Error de Base de Datos: {e}")